In [0]:
%run ./00_config_and_helpers

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

dbutils.widgets.text("ingest_date", "*")        # * = all, or e.g. 2026-10-09
dbutils.widgets.text("bronze_batch_id", "*")    # * = all, or a specific Bronze batch
ING = dbutils.widgets.get("ingest_date")
BB  = dbutils.widgets.get("bronze_batch_id")

try:
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
except Exception:
    pass

def read_bronze(table):
    df = spark.table(table)
    if ING != "*": df = df.where(F.col("_ingest_date") == ING)
    if BB  != "*": df = df.where(F.col("_batch_id") == BB)
    return df

def dedupe(df, keys):
 # remove duplicates    w = Window.partitionBy(*keys).orderBy(F.col("_bronze_ts").desc())
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")

def finalize(df):
 # ready them for write to silver    biz = [c for c in df.columns if c not in ("_bronze_ts", "_bad")]
    return (df.withColumn("_row_hash", F.sha2(F.to_json(F.struct(*biz)), 256))
              .withColumn("load_timestamp", F.current_timestamp())
              .withColumn("_batch_id", F.lit(BATCH_ID))
              .drop("_bronze_ts", "_bad"))

def quarantine(bad_df, entity, key_col):
 # for bad rows    start = datetime.now()
    q = bad_df.select(F.lit(entity).alias("entity"),
                      F.col(key_col).cast("string").alias("record_key"),
                      F.col("repo_name"),
                      F.lit("null key or timestamp cast failed").alias("reason"),
                      F.current_timestamp().alias("load_timestamp"),
                      F.lit(BATCH_ID).alias("_batch_id"))
    q.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("silver_quarantine")

def merge_into(df, table, keys, param):
    start = datetime.now()
    try:
        n_in = df.count()
        if not spark.catalog.tableExists(table):
            df.write.format("delta").saveAsTable(table)
            ins, upd = n_in, 0
        else:
            tgt = DeltaTable.forName(spark, table)
            cond = " AND ".join([f"t.{k} <=> s.{k}" for k in keys])
            (tgt.alias("t").merge(df.alias("s"), cond)
                .whenMatchedUpdateAll(condition="t._row_hash <> s._row_hash")
                .whenNotMatchedInsertAll()
                .execute())
            m = tgt.history(1).collect()[0]["operationMetrics"]
            ins = int(m.get("numTargetRowsInserted", 0))
            upd = int(m.get("numTargetRowsUpdated", 0))
        log_run("Bronze-to-Silver", f"{table} <- {param}", start, "Success", ins, upd)
        print(f"{table}: {ins} inserted, {upd} updated (input {n_in})")
    except Exception as e:
        log_run("Bronze-to-Silver", f"{table} <- {param}", start, "Failure", error=str(e)[:500])
        print(f"FAILED {table}: {e}")

param = f"ingest_date={ING}, bronze_batch_id={BB}"

# ---------------- silver_commits ----------------
b = read_bronze("bronze_commits")
c = b.select(
    F.col("sha"),
    F.col("_source_repo").alias("repo_name"),
    F.to_timestamp("commit.author.date").alias("author_ts"),
    F.to_timestamp("commit.committer.date").alias("committer_ts"),
    F.col("commit.author.name").alias("author_name"),
    F.col("author.login").alias("author_login"),
    F.when(F.col("commit.author.email").isNotNull(),
           F.sha2(F.lower(F.trim(F.col("commit.author.email"))), 256)).alias("contributor_hash"),
    F.col("commit.message").alias("commit_message"),
    F.col("commit.comment_count").cast("int").alias("comment_count"),
    F.greatest(F.size("parents"), F.lit(0)).alias("parent_count"),
    F.col("commit.verification.verified").alias("is_verified"),
    F.col("html_url"),
    F.col("load_timestamp").alias("_bronze_ts"),
    (F.col("sha").isNull() |
     (F.col("commit.author.date").isNotNull() & F.to_timestamp("commit.author.date").isNull())).alias("_bad"),
).withColumn("event_date", F.to_date("author_ts"))

quarantine(c.where("_bad"), "commits", "sha")
commits = finalize(dedupe(c.where("NOT _bad"), ["repo_name", "sha"]))
merge_into(commits, "silver_commits", ["repo_name", "sha"], param)

# ---------------- silver_issues ----------------
i = read_bronze("bronze_issues")
iss = i.select(
    F.col("id").alias("issue_id"),
    F.col("_source_repo").alias("repo_name"),
    F.col("number"),
    F.col("state"),
    F.col("title"),
    F.col("body"),
    F.col("user.login").alias("author_login"),
    F.when(F.col("user.login").isNotNull(),
           F.sha2(F.lower(F.col("user.login")), 256)).alias("author_hash"),
    F.col("pull_request.url").isNotNull().alias("is_pull_request"),
    F.col("pull_request.merged_at").isNotNull().alias("is_merged"),
    F.col("comments").cast("int").alias("comment_count"),
    F.to_timestamp("created_at").alias("created_ts"),
    F.to_timestamp("updated_at").alias("updated_ts"),
    F.to_timestamp("closed_at").alias("closed_ts"),
    F.col("author_association"),
    F.col("html_url"),
    F.col("load_timestamp").alias("_bronze_ts"),
    (F.col("id").isNull() |
     (F.col("created_at").isNotNull() & F.to_timestamp("created_at").isNull())).alias("_bad"),
).withColumn("event_date", F.to_date("created_ts"))

quarantine(iss.where("_bad"), "issues", "issue_id")
issues = finalize(dedupe(iss.where("NOT _bad"), ["repo_name", "issue_id"]))
merge_into(issues, "silver_issues", ["repo_name", "issue_id"], param)

# ---------------- silver_issue_labels ----------------
lab = (i.select(F.col("id").alias("issue_id"),
                F.col("_source_repo").alias("repo_name"),
                F.explode("labels").alias("l"),
                F.col("load_timestamp").alias("_bronze_ts"))
         .select("issue_id", "repo_name",
                 F.col("l.id").alias("label_id"),
                 F.col("l.name").alias("label_name"),
                 F.col("l.color").alias("label_color"),
                 "_bronze_ts")
         .withColumn("_bad", F.lit(False)))
labels = finalize(dedupe(lab, ["repo_name", "issue_id", "label_name"]))
merge_into(labels, "silver_issue_labels", ["repo_name", "issue_id", "label_name"], param)

In [0]:
display(spark.sql("SELECT repo_name, COUNT(*) AS rows FROM silver_commits GROUP BY 1"))
display(spark.sql("SELECT repo_name, COUNT(*) AS rows FROM silver_issues GROUP BY 1"))
display(spark.sql("SELECT * FROM silver_commits LIMIT 5"))
display(spark.sql("SELECT layer, parameter_or_file, status, rows_inserted, rows_updated FROM pipeline_execution_logs WHERE layer='Bronze-to-Silver' ORDER BY start_time DESC LIMIT 6"))